# Brain Tumor Detection with YOLOv11
This notebook trains, validates, tests, and exports a YOLOv11 model for Astrocytoma and Glioblastoma detection.

> **Medical disclaimer:** Coursework use only. This model is not suitable for clinical diagnosis.

In [ ]:
!nvidia-smi
!pip -q install ultralytics>=8.3.0
from google.colab import files
from pathlib import Path
import os, random, shutil
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
print('Ultralytics installed successfully.')

In [ ]:
# Upload brain.yolov11.zip when prompted.
uploaded = files.upload()
zip_name = next(iter(uploaded))
!rm -rf /content/brain_dataset
!unzip -q "$zip_name" -d /content/brain_dataset
DATA_ROOT = Path('/content/brain_dataset')
yaml_files = list(DATA_ROOT.rglob('data.yaml'))
assert yaml_files, 'data.yaml was not found after extraction.'
DATA_YAML = yaml_files[0]
print(DATA_YAML)
print(DATA_YAML.read_text())

In [ ]:
# Make dataset paths absolute so Ultralytics can find every split reliably.
import yaml
config = yaml.safe_load(DATA_YAML.read_text())
for split in ('train', 'val', 'test'):
    if split in config:
        config[split] = str((DATA_YAML.parent / config[split]).resolve())
ABS_DATA_YAML = Path('/content/brain_data_absolute.yaml')
ABS_DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
print(ABS_DATA_YAML.read_text())

for split in ('train', 'valid', 'test'):
    image_dir = DATA_YAML.parent / split / 'images'
    label_dir = DATA_YAML.parent / split / 'labels'
    print(f'{split}: {len(list(image_dir.glob("*")))} images, {len(list(label_dir.glob("*.txt")))} labels')

In [ ]:
# Train YOLOv11 nano. Increase epochs to 100+ if GPU time allows.
model = YOLO('yolo11n.pt')
results = model.train(
    data=str(ABS_DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    optimizer='auto',
    project='/content/runs',
    name='brain_yolo11n',
    seed=42,
    plots=True
)
BEST_WEIGHTS = Path('/content/runs/brain_yolo11n/weights/best.pt')
assert BEST_WEIGHTS.exists(), 'Training did not create best.pt'
print(BEST_WEIGHTS)

In [ ]:
# Validate using the held-out validation split and show learning curves.
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(data=str(ABS_DATA_YAML), imgsz=640, split='val', plots=True)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)
display(Image.open('/content/runs/brain_yolo11n/results.png'))

In [ ]:
# Final evaluation on the untouched test split.
test_metrics = best_model.val(data=str(ABS_DATA_YAML), imgsz=640, split='test', plots=True)
print('Test mAP50:', test_metrics.box.map50)
print('Test mAP50-95:', test_metrics.box.map)

test_images = list((DATA_YAML.parent / 'test' / 'images').glob('*'))
sample_image = random.choice(test_images)
prediction = best_model.predict(source=str(sample_image), conf=0.25, save=True)
print('Predicted image:', sample_image)
display(Image.open(prediction[0].save_dir / sample_image.name))

In [ ]:
# Download trained weights and result plots for the app and assignment report.
files.download(str(BEST_WEIGHTS))
files.download('/content/runs/brain_yolo11n/results.png')
# Optional: export to ONNX for deployment.
best_model.export(format='onnx', imgsz=640)